In [1]:
import pandas as pd
import sqlite3

# Create a SQLite database file
conn = sqlite3.connect('../data/ecommerce.db')

# Load cleaned CSVs into the database as tables
master = pd.read_csv('../data/cleaned/master_orders.csv')
monthly = pd.read_csv('../data/cleaned/monthly_summary.csv')
category = pd.read_csv('../data/cleaned/category_summary.csv')
state = pd.read_csv('../data/cleaned/state_summary.csv')

master.to_sql('master_orders', conn, if_exists='replace', index=False)
monthly.to_sql('monthly_summary', conn, if_exists='replace', index=False)
category.to_sql('category_summary', conn, if_exists='replace', index=False)
state.to_sql('state_summary', conn, if_exists='replace', index=False)

print("Tables loaded into ecommerce.db!")

# Verify
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in database:", cursor.fetchall())

Tables loaded into ecommerce.db!
Tables in database: [('master_orders',), ('monthly_summary',), ('category_summary',), ('state_summary',)]


In [2]:
query1 = """
SELECT 
    product_category_name_english AS category,
    COUNT(DISTINCT order_id)      AS total_orders,
    ROUND(SUM(price), 2)          AS total_revenue,
    ROUND(AVG(price), 2)          AS avg_order_value
FROM master_orders
WHERE product_category_name_english != 'unknown'
GROUP BY product_category_name_english
ORDER BY total_revenue DESC
LIMIT 10;
"""

result1 = pd.read_sql_query(query1, conn)
print("=== TOP 10 CATEGORIES BY REVENUE ===")
print(result1.to_string(index=False))

=== TOP 10 CATEGORIES BY REVENUE ===
             category  total_orders  total_revenue  avg_order_value
        health_beauty          8647     1233131.72           130.28
        watches_gifts          5495     1166176.98           199.04
       bed_bath_table          9272     1023434.76            93.44
       sports_leisure          7530      954852.55           113.25
computers_accessories          6530      888724.61           116.26
      furniture_decor          6307      711927.69            87.25
           housewares          5743      615628.69            90.60
           cool_stuff          3559      610204.10           164.12
                 auto          3810      578966.65           139.85
                 toys          3804      471286.48           116.94


In [3]:
query2 = """
SELECT
    order_year_month,
    COUNT(DISTINCT order_id)   AS total_orders,
    ROUND(SUM(price), 2)       AS total_revenue,
    ROUND(AVG(price), 2)       AS avg_order_value
FROM master_orders
GROUP BY order_year_month
ORDER BY order_year_month;
"""

result2 = pd.read_sql_query(query2, conn)
print("=== MONTHLY REVENUE TREND ===")
print(result2.to_string(index=False))

=== MONTHLY REVENUE TREND ===
order_year_month  total_orders  total_revenue  avg_order_value
         2016-09             1         134.97            44.99
         2016-10           265       40325.11           128.83
         2016-12             1          10.90            10.90
         2017-01           750      111798.36           122.45
         2017-02          1653      234223.40           126.06
         2017-03          2546      359198.85           123.99
         2017-04          2303      340669.68           132.61
         2017-05          3546      489338.25           122.21
         2017-06          3135      421923.37           120.93
         2017-07          3872      481604.52           109.06
         2017-08          4193      554699.70           115.63
         2017-09          4150      607399.67           128.22
         2017-10          4478      648247.65           124.33
         2017-11          7289      987765.37           116.55
         2017-12         

In [4]:
query3 = """
SELECT
    customer_state                          AS state,
    COUNT(DISTINCT order_id)               AS total_orders,
    ROUND(SUM(price), 2)                   AS total_revenue,
    ROUND(AVG(price), 2)                   AS avg_order_value,
    ROUND(AVG(delivery_days), 1)           AS avg_delivery_days,
    ROUND(
        SUM(CASE WHEN delivered_on_time = 1 THEN 1 ELSE 0 END) * 100.0
        / COUNT(*), 1
    )                                      AS on_time_pct
FROM master_orders
GROUP BY customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

result3 = pd.read_sql_query(query3, conn)
print("=== STATE PERFORMANCE ===")
print(result3.to_string(index=False))

=== STATE PERFORMANCE ===
state  total_orders  total_revenue  avg_order_value  avg_delivery_days  on_time_pct
   SP         40501     5067633.16           109.10                8.3         94.2
   RJ         12350     1759651.13           124.42               14.7         87.0
   MG         11354     1552481.83           120.20               11.5         94.6
   RS          5345      728897.47           118.83               14.7         93.1
   PR          4923      666063.51           117.91               11.5         95.2
   SC          3546      507012.13           123.75               14.5         90.4
   BA          3256      493584.14           134.02               18.8         86.3
   DF          2080      296498.41           125.90               12.5         92.6
   GO          1957      282836.70           124.21               14.9         92.1
   ES          1995      268643.45           120.74               15.2         87.8


In [5]:
query4 = """
WITH delivery_summary AS (
    SELECT
        order_id,
        delivery_days,
        delivered_on_time,
        CASE
            WHEN delivery_days <= 7  THEN 'Fast (0-7 days)'
            WHEN delivery_days <= 14 THEN 'Normal (8-14 days)'
            WHEN delivery_days <= 21 THEN 'Slow (15-21 days)'
            ELSE 'Very slow (21+ days)'
        END AS delivery_bucket
    FROM master_orders
    WHERE delivery_days IS NOT NULL
      AND delivery_days >= 0
)
SELECT
    delivery_bucket,
    COUNT(DISTINCT order_id)  AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1
    )                         AS pct_of_orders
FROM delivery_summary
GROUP BY delivery_bucket
ORDER BY MIN(delivery_days);
"""

result4 = pd.read_sql_query(query4, conn)
print("=== DELIVERY SPEED BREAKDOWN ===")
print(result4.to_string(index=False))

=== DELIVERY SPEED BREAKDOWN ===
     delivery_bucket  total_orders  pct_of_orders
     Fast (0-7 days)         33696           35.1
  Normal (8-14 days)         36397           37.9
   Slow (15-21 days)         15369           15.9
Very slow (21+ days)         11008           11.1


In [6]:
query5 = """
WITH category_totals AS (
    SELECT
        product_category_name_english           AS category,
        ROUND(SUM(price), 2)                    AS category_revenue
    FROM master_orders
    WHERE product_category_name_english != 'unknown'
    GROUP BY product_category_name_english
),
grand_total AS (
    SELECT SUM(price) AS total FROM master_orders
)
SELECT
    c.category,
    c.category_revenue,
    ROUND(c.category_revenue * 100.0 / g.total, 2) AS revenue_share_pct
FROM category_totals c, grand_total g
ORDER BY c.category_revenue DESC
LIMIT 10;
"""

result5 = pd.read_sql_query(query5, conn)
print("=== CATEGORY REVENUE SHARE ===")
print(result5.to_string(index=False))

conn.close()
print("\nDatabase connection closed.")

=== CATEGORY REVENUE SHARE ===
             category  category_revenue  revenue_share_pct
        health_beauty        1233131.72               9.33
        watches_gifts        1166176.98               8.82
       bed_bath_table        1023434.76               7.74
       sports_leisure         954852.55               7.22
computers_accessories         888724.61               6.72
      furniture_decor         711927.69               5.38
           housewares         615628.69               4.66
           cool_stuff         610204.10               4.62
                 auto         578966.65               4.38
                 toys         471286.48               3.56

Database connection closed.


In [2]:
import os

sql_content = """-- ============================================
-- E-Commerce Sales Analysis — SQL Queries
-- Dataset: Brazilian E-Commerce (Olist)
-- Author: Swetha G
-- ============================================

-- Query 1: Top 10 categories by revenue
SELECT 
    product_category_name_english AS category,
    COUNT(DISTINCT order_id)      AS total_orders,
    ROUND(SUM(price), 2)          AS total_revenue,
    ROUND(AVG(price), 2)          AS avg_order_value
FROM master_orders
WHERE product_category_name_english != 'unknown'
GROUP BY product_category_name_english
ORDER BY total_revenue DESC
LIMIT 10;

-- -----------------------------------------------

-- Query 2: Monthly revenue and order trend
SELECT
    order_year_month,
    COUNT(DISTINCT order_id)   AS total_orders,
    ROUND(SUM(price), 2)       AS total_revenue,
    ROUND(AVG(price), 2)       AS avg_order_value
FROM master_orders
GROUP BY order_year_month
ORDER BY order_year_month;

-- -----------------------------------------------

-- Query 3: Revenue and delivery performance by state
SELECT
    customer_state                          AS state,
    COUNT(DISTINCT order_id)               AS total_orders,
    ROUND(SUM(price), 2)                   AS total_revenue,
    ROUND(AVG(price), 2)                   AS avg_order_value,
    ROUND(AVG(delivery_days), 1)           AS avg_delivery_days,
    ROUND(
        SUM(CASE WHEN delivered_on_time = 1 THEN 1 ELSE 0 END) * 100.0
        / COUNT(*), 1
    )                                      AS on_time_pct
FROM master_orders
GROUP BY customer_state
ORDER BY total_revenue DESC
LIMIT 10;

-- -----------------------------------------------

-- Query 4: Delivery speed breakdown using CTE
WITH delivery_summary AS (
    SELECT
        order_id,
        delivery_days,
        delivered_on_time,
        CASE
            WHEN delivery_days <= 7  THEN 'Fast (0-7 days)'
            WHEN delivery_days <= 14 THEN 'Normal (8-14 days)'
            WHEN delivery_days <= 21 THEN 'Slow (15-21 days)'
            ELSE 'Very slow (21+ days)'
        END AS delivery_bucket
    FROM master_orders
    WHERE delivery_days IS NOT NULL
      AND delivery_days >= 0
)
SELECT
    delivery_bucket,
    COUNT(DISTINCT order_id)  AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1
    )                         AS pct_of_orders
FROM delivery_summary
GROUP BY delivery_bucket
ORDER BY MIN(delivery_days);

-- -----------------------------------------------

-- Query 5: Category revenue share (% of total)
WITH category_totals AS (
    SELECT
        product_category_name_english           AS category,
        ROUND(SUM(price), 2)                    AS category_revenue
    FROM master_orders
    WHERE product_category_name_english != 'unknown'
    GROUP BY product_category_name_english
),
grand_total AS (
    SELECT SUM(price) AS total FROM master_orders
)
SELECT
    c.category,
    c.category_revenue,
    ROUND(c.category_revenue * 100.0 / g.total, 2) AS revenue_share_pct
FROM category_totals c, grand_total g
ORDER BY c.category_revenue DESC
LIMIT 10;
"""

# Windows-safe path
sql_path = os.path.join(os.getcwd(), '..', 'sql', 'analysis_queries.sql')
sql_path = os.path.abspath(sql_path)

# Create the folder if it doesn't exist
os.makedirs(os.path.dirname(sql_path), exist_ok=True)

with open(sql_path, 'w') as f:
    f.write(sql_content)

print(f"SQL file saved to: {sql_path}")

SQL file saved to: C:\Users\sweth\ecommerce-sales-analysis\sql\analysis_queries.sql
